# Bike Availability Forecasting

This notebook is just making a naive forecasting based on the average of the data.

In [15]:
import numpy as np
import xarray as xr

# Tensor decomposition
from tensorly.decomposition import tucker

from utilities import To_Numpy, concat_features

np.random.seed(42)

LOAD DATA

In [16]:
# Load dataset
ds = xr.open_dataset("../full_db.nc")
ds["bike"] = ds["vm_disp"] + ds["vae_disp"]

print(ds)

<xarray.Dataset> Size: 357MB
Dimensions:          (station: 1551, binary_index: 11, spatial_feature: 4,
                      time: 17544, ferie_feature: 2, component: 10, meteo_dim: 8)
Coordinates:
  * station          (station) <U12 74kB '000000002' '10001' ... '9999980'
    capacite         (station) int32 6kB ...
    latitude         (station) float64 12kB ...
    longitude        (station) float64 12kB ...
  * binary_index     (binary_index) int32 44B 0 1 2 3 4 5 6 7 8 9 10
  * spatial_feature  (spatial_feature) <U12 192B 'Capacité de ' ... 'altitude'
  * time             (time) datetime64[ns] 140kB 2023-01-01 ... 2024-12-31T23...
  * ferie_feature    (ferie_feature) <U8 64B 'vacances' 'is_ferie'
  * component        (component) int64 80B 0 1 2 3 4 5 6 7 8 9
  * meteo_dim        (meteo_dim) <U5 160B 'DRR1' 'FXI' 'GLO' ... 'TX' 'U'
Data variables: (12/20)
    code_binaire     (station, binary_index) int32 68kB ...
    features         (station, spatial_feature) float64 50kB ...
   

In [21]:
print(ds.features)

<xarray.DataArray 'features' (station: 1551, spatial_feature: 4)> Size: 50kB
[6204 values with dtype=float64]
Coordinates:
  * station          (station) <U12 74kB '000000002' '10001' ... '9999980'
    capacite         (station) int32 6kB 16 63 0 19 49 18 30 ... 43 30 23 38 0 0
    latitude         (station) float64 12kB ...
    longitude        (station) float64 12kB ...
  * spatial_feature  (spatial_feature) <U12 192B 'Capacité de ' ... 'altitude'


In [22]:
print(ds.bike)

<xarray.DataArray 'bike' (station: 1551, time: 17544)> Size: 27MB
array([[ 0,  0,  0, ...,  0,  0,  0],
       [12,  8, 10, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       ...,
       [34, 33, 33, ..., 10, 10,  9],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0]], shape=(1551, 17544), dtype=uint8)
Coordinates:
  * station    (station) <U12 74kB '000000002' '10001' ... '9999970' '9999980'
    capacite   (station) int32 6kB 16 63 0 19 49 18 30 19 ... 21 43 30 23 38 0 0
    latitude   (station) float64 12kB ...
    longitude  (station) float64 12kB ...
  * time       (time) datetime64[ns] 140kB 2023-01-01 ... 2024-12-31T23:00:00


In [43]:
to_print= ds.bike
print(to_print)

<xarray.DataArray 'bike' (station: 1551, time: 17544)> Size: 27MB
array([[ 0,  0,  0, ...,  0,  0,  0],
       [12,  8, 10, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0],
       ...,
       [34, 33, 33, ..., 10, 10,  9],
       [ 0,  0,  0, ...,  0,  0,  0],
       [ 0,  0,  0, ...,  0,  0,  0]], shape=(1551, 17544), dtype=uint8)
Coordinates:
  * station    (station) <U12 74kB '000000002' '10001' ... '9999970' '9999980'
    capacite   (station) int32 6kB 16 63 0 19 49 18 30 19 ... 21 43 30 23 38 0 0
    latitude   (station) float64 12kB ...
    longitude  (station) float64 12kB ...
  * time       (time) datetime64[ns] 140kB 2023-01-01 ... 2024-12-31T23:00:00


TIME CONFIGURATION

In [49]:
# First Monday of 2023 (dataset starts on a Sunday)
first_monday_index = 24

n_stations = len(ds["station"])

n_time_full = len(ds["time"]) - first_monday_index

n_weeks = n_time_full // (7 * 24)

n_time = 24 * 7 * n_weeks
n_days = 7 * n_weeks

# ---------------------------------------------------------
# Static covariates
# ---------------------------------------------------------
#memory efficiency
float_tp = np.float32
#Xarray -> Numpy
to_numpy = To_Numpy(ds, first_monday_index, n_weeks, float_tp)

#/!\ a few station capacity are negative due to a preprocessing bug
capacity = np.positive(to_numpy("capacite"))

lat = to_numpy("latitude")
lon = to_numpy("longitude")
#print(lat.shape, lon.shape)
Meteo = to_numpy("meteo_h")
Holiday = to_numpy("feries")
#print(Meteo.shape, Holiday.shape)
#print(Meteo[:2,:2])
#print(Holiday[:2,:2])

IMPORTANT: EMBEDDING FUNCTION (with Tucker)

In [62]:
rank = [10, n_weeks, 7, 10]

def tensor_encoding(x):
    """
    Extract spatial and temporal embeddings using Tucker decomposition.

    Dimensions:
    station × week × day × hour
    """

    _, factors = tucker(
        x.reshape(n_stations, n_weeks, 7, 24).astype(np.float32),
        rank=rank
    )

    spatial_encoding = (factors[0] / (capacity[:, None] + 0.01))
    spatial_features = concat_features([capacity[:, None], spatial_encoding]).astype(float_tp)

    day_encoding = factors[2]
    hour_encoding = factors[3]

    time_encoding = concat_features(
        [day_encoding[:, None, :], hour_encoding[None, :, :]],
        shape=[7, 24],
        flatten=True
    )
    temporal_features = time_encoding.astype(float_tp)

    return spatial_features, temporal_features

In [59]:
def prepare_data(variable):
    """
    Prepare base tensors and embeddings for a given variable.
    """

    Y = to_numpy(variable)

    spatial_features, temporal_features = tensor_encoding(Y)

    spatial_features = np.concatenate(
        [spatial_features, lat[:, None], lon[:, None]],
        axis=1
    )

    return {
        "Y": Y,
        "space": spatial_features,
        "time": temporal_features
    }

PREPARE DATA FOR REGRESSION TASK

In [63]:
#prepare target embedding (Tucker)
data_dict = prepare_data("bike")

In [67]:
print(data_dict["Y"].shape)
print(data_dict["space"].shape)
print(data_dict["time"].shape)

(1551, 104, 168)
(1551, 13)
(168, 17)


In [75]:
Y = to_numpy("bike")
T = 8

In [76]:
print(Y.shape)
print(n_stations, n_weeks, n_time//n_weeks)

(1551, 104, 168)
1551 104 168


In [ ]:
def in_week_end(t):
    return int(t/24)%7>5

def pred_for_time(T, Y):
    for week in Y.shape[1]:
        for hour in Y.shape[2]:
            if in_week_end(hour):
                y_to_avg=Y[:,week-T:week,hour:hour+24] # if saturday, not robust
            else :
                break #TODO for week
    return np.mean(y_to_avg)

def rmse(y, pred):
    return np.sqrt(np.mean((pred-y)**2))

(1551, 10)
(104, 104)
(7, 7)
(24, 10)
4


PREDICTION FUNCTIONS